In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/several27/FakeNewsCorpus/master/news_sample.csv"
# Kept for testing
file_name = "995,000_rows.csv"
df = pd.read_csv(file_name)
df["content"] = df["content"].fillna("").astype(str) # Otherwise pandas introduces float errors.

df.head()

In [ ]:
from joblib import Parallel, delayed
import nltk
from tqdm.notebook import tqdm

texts = df["content"].astype(str).tolist()

tokens = [nltk.word_tokenize(t) for t in tqdm(texts)]

del df["content"]
df["tokens"] = tokens
df.to_pickle("df_tokens.pkl") # Save data first


In [ ]:
import nltk
import string
from nltk.corpus import stopwords
from tqdm.notebook import tqdm

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
punctuation = set(string.punctuation)

def clean_tokens(tokens):
    return [word for word in tokens
            if word.lower() not in stop_words
            and word not in punctuation
            and word.isalpha()]

df["tokens_clean"] = [clean_tokens(tokens) for tokens in tqdm(df["tokens"])]

print(df["tokens_clean"])


In [ ]:
from nltk.stem import PorterStemmer
from tqdm.notebook import tqdm

stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(w) for w in tokens]

df["tokens_clean_stemmed"] = [stem_tokens(tokens) for tokens in tqdm(df["tokens_clean"])]

print(df["tokens_clean_stemmed"])
print(df["tokens_clean_stemmed"].apply(len).sum())

In [ ]:
total_with_stopwords = df["tokens"].apply(len).sum()
print(f"Tokens before removing stopwords: {total_with_stopwords}")
total_no_stopwords = df["tokens_clean_stemmed"].apply(len).sum()
print(f"Tokens after removing stopwords: {total_no_stopwords}")
reduction_rate = (total_with_stopwords - total_no_stopwords) / total_with_stopwords
print(f"reduction rate in percent: {reduction_rate * 100}")

In [ ]:
df.to_csv("processed.csv", index=False)

In [ ]:
import re 
import csv 
import matplotlib.pyplot as plt

with open("output.txt", newline="") as file:
    reader = csv.reader(file)
    data = "\n".join([", ".join(row) for row in reader])

def countNum(data):
    hash = {}
    pattern = "[^ |,|\t|\n|:|-|\.|\?|\!]+"
    match = re.findall(pattern, data)

    for i in match: 
        if i not in hash:
            hash[i] = 1
        else:
            hash[i] += 1
    return hash

def sortlist(d):
    d = d.copy()  
    sorted_list = []

    while d:
        max_key = max(d, key=d.get)
        sorted_list.append((d[max_key], max_key))
        del d[max_key]

    return sorted_list

# Make the untoched data fit the function and then sort the frequency
all_tokens = [word for tokens in df["tokens"] for word in tokens]
all_tokens_string = " ".join(all_tokens) 
top_words_default = sortlist(countNum(all_tokens_string))[:100]

# Sort the data from the output.txt file which has been cleaned 
top_words_cleaned = sortlist(countNum(data))[:100]



words_default = [w for w, c in top_words_default]
counts_default = [c for w, c in top_words_default]

words_cleaned = [w for w, c in top_words_cleaned]
counts_cleaned = [c for w, c in top_words_cleaned]

# Plot the bar chart
plt.figure(figsize=(15, 6))  
plt.bar(counts_cleaned, words_cleaned, color='skyblue')
plt.xticks(rotation=90)      
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.title('Top 100 Most Frequent Words - Cleaned')
plt.tight_layout()
plt.show()

# Plot the bar chart
plt.figure(figsize=(15, 6))  
plt.bar(counts_default, words_default, color='skyblue')
plt.xticks(rotation=90)      
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.title('Top 100 Most Frequent Words - Default')
plt.tight_layout()
plt.show()

#Find out how many types there are.
unique_types = df["type"].unique()
for article_type in unique_types:
    print(article_type)
print(f"How many unique types are there?: {len(unique_types)}")

unknown = df[df["type"] == "unknown"]
nan = df[df["type"] == "nan"]
df["type"] = df["type"].str.strip()  # remove any surrounding whitespace
nan = df[df["type"] == "nan"]
print(f"There are {len(nan)} nan type articles, and {len(unknown)} unknown articles")